# ML-10 -- Content Action Playbook

**FL-07 deliverable (`w07_action_playbook.ipynb`).** This notebook turns the validated W05/W06 logistic
model into a **content action playbook**: an honest, reason-coded, ranked queue of "what a content person
should look at first" -- plus the rules for using it, its limits, a human-review + no-go list, cost/value
thinking, monitoring / retrain triggers, and the exports for the paper.

**Evidence rules (from `skills/writing-honest-claims` + `skills/flyrank/flyrank-data`):**
- Every number is measured in this run from `data/raw/content_refresh_anonymized.csv`, or is explicitly labeled **proposed / governance**.
- **Rate columns are x100 percentages** (`ctr = 0.76` means 0.76%); `avg_position = 0` means "no data" -- never rank zero.
- `trend_direction` / `trend_pct` construct the label (`decline_label`) and are **never** model features.
- Wording ladder: *observed / measured / directional / decision-support*. No causal claims; refresh effects are **not** measured anywhere in this repo.

## 1. Ranked actions + reason codes

*The head of a content queue, not a passive list.*

The model's job is decision support: it ranks all 30,000 pseudonymous content items by out-of-fold
probability of the observed "declining" label, and we attach the **reason** each item is ranked where it
is in words a reviewer can re-derive from the raw metrics. A content editor opens the queue, starts at
rank 1, and acts only after a human check (Section 3). Measured base rate of declining in this snapshot:
**0.542** -- that is the "random" reference for every precision number below.

In [1]:
# --- 1a. Score the whole portfolio with the validated recipe (out-of-fold, honestly) ------
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

DATA = Path("../../data/raw/content_refresh_anonymized.csv")
assert DATA.exists(), "starter dataset missing"
df = pd.read_csv(DATA)
df["decline_label"] = (df["trend_direction"] == "down").astype(int)

CATEGORICAL_FEATURES = ["content_type", "main_intent"]
NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc",
    "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d",
    "sessions_90d", "users_90d", "engaged_sessions_90d",
    "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
assert len(FEATURES) == 24 and len(set(FEATURES)) == 24, "24 unique features"
forbidden = {"content_id", "client_id", "trend_direction", "trend_pct", "decline_label"}
assert set(FEATURES).isdisjoint(forbidden), "a label column leaked into features"

def build_pipeline():
    num = Pipeline([("imp", SimpleImputer(strategy="median")), ("scl", StandardScaler())])
    cat = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                    ("ohe", OneHotEncoder(handle_unknown="ignore"))])
    pre = ColumnTransformer([("num", num, NUMERIC_FEATURES), ("cat", cat, CATEGORICAL_FEATURES)])
    return Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000, random_state=42))])

def pct_at_k(yt, pr, k):
    """Precision at k = share of top-k that are truly declining (label is in this snapshot)."""
    if len(yt) < k:
        return np.nan
    return round(float(np.mean(yt[np.argsort(pr)[::-1][:k]])), 3)

X = df[FEATURES]; y = df["decline_label"].to_numpy(); G = df["client_id"].to_numpy()

# Out-of-fold: one model per client-grouped fold; each row is scored only by the fold that held
# ITS client out. This is the exact validation design from the W06 audit.
oof = np.full(len(df), np.nan)
fold_rows = []
for i, (tr, va) in enumerate(GroupKFold(n_splits=5).split(X, y, groups=G)):
    assert set(G[tr]).isdisjoint(G[va]), "client leaked across folds"
    m = build_pipeline().fit(X.iloc[tr], y[tr])
    pv = m.predict_proba(X.iloc[va])[:, 1]
    oof[va] = pv
    yv = y[va]
    fold_rows.append({"fold": i + 1,
                      "n_test": int(len(yv)),
                      "base": round(float(yv.mean()), 3),
                      "P@20": pct_at_k(yv, pv, 20),
                      "P@50": pct_at_k(yv, pv, 50),
                      "P@100": pct_at_k(yv, pv, 100),
                      "AUC": round(float(roc_auc_score(yv, pv)), 3)})
folds = pd.DataFrame(fold_rows)
folds

df["model_probability"] = oof
assert np.isfinite(oof).all(), "every row must have an out-of-fold score"
print("AUC mean +/- sd across folds : %.3f +/- %.3f" % (folds["AUC"].mean(), folds["AUC"].std()))
print("P@50   mean +/- sd across folds: %.3f +/- %.3f" % (folds["P@50"].mean(), folds["P@50"].std()))
print("pooled   base rate             : %.3f" % float(y.mean()))

AUC mean +/- sd across folds : 0.632 +/- 0.037
P@50   mean +/- sd across folds: 0.708 +/- 0.148
pooled   base rate             : 0.542


### Reason codes (the "why" on the queue)

Every row gets the codes whose **data conditions** hold on it -- transparent, re-derivable rules:

| Reason code | Condition (measured columns in this dataset) |
|---|---|
| `stale_visible_page` | `days_since_last_update >= 180` **and** `impressions_90d >= 500` |
| `declining_with_demand` | label = down **and** `impressions_90d >= 100` (still visible while declining) |
| `thin_visible_page` | `0 < word_count < 1200` **and** `impressions_90d >= 250` |
| `page_one_decay_risk` | `avg_position in (0,10]` **and** `content_age_days >= 180` |
| `low_ctr_visible_page` | `impressions >= 500`, `position <= 20`, `ctr < 0.5%` |
| `low_engagement_visible_page` | `sessions >= 30` (**and** `engagement_rate < 30%` or `scroll_rate < 30%`) |
| `model_decline_risk` | out-of-fold `P(decline) >= 0.50` (**proposed threshold**) on the validated logistic model |
| `no_position_data` / `no_keyword_data` / `no_wordcount` | the metric that would justify reasoning is missing -- a reason to **not** act, not to act |

**Archetype -> action** (each row gets exactly one archetype). `priority` is a **proposed / heuristic**
3-level label **cut from the model score, not validated**: `high >= 0.65`, `medium >= 0.50`, else `low`.
The 0.65 / 0.50 cuts are operational thresholds chosen by us; no claim that they are optimal or measured.

* `refresh_candidate` -> **refresh**
* `thin_content_candidate` -> **expand_and_refresh**
* `ctr_candidate` -> **refresh_and_review_ctr**
* `engagement_candidate` -> **refresh_and_review_engagement**
* `ranking_candidate` -> **recheck_position** (visible but below page one, worth a positioning review)
* `monitor_candidate` -> **monitor**
* `insufficient_data` -> **human_review** (missing keyword/position evidence -- never automated)
* `healthy` -> **monitor**

In [2]:
# ================= 1b. Build the ranked action queue (scores + reason codes + actions) ======
q = df.copy()
q["has_kw"] = q["search_volume"].notna()
q["has_wc"] = q["word_count"].notna()

REASON_LIST = []
for _, r in q.iterrows():
    rc = []
    if r["days_since_last_update"] >= 180 and r["impressions_90d"] >= 500:
        rc.append("stale_visible_page")
    if r["trend_direction"] == "down" and r["impressions_90d"] >= 100:
        rc.append("declining_with_demand")
    if r["has_wc"] and 0 < r["word_count"] < 1200 and r["impressions_90d"] >= 250:
        rc.append("thin_visible_page")
    if r["avg_position"] > 0 and r["avg_position"] <= 10 and r["content_age_days"] >= 180:
        rc.append("page_one_decay_risk")
    if r["impressions_90d"] >= 500 and 0 < r["avg_position"] <= 20 and r["ctr"] < 0.5:
        rc.append("low_ctr_visible_page")
    if r["sessions_90d"] >= 30 and (0 < r["engagement_rate"] < 30 or 0 < r["scroll_rate"] < 30):
        rc.append("low_engagement_visible_page")
    if r["model_probability"] >= 0.50:
        rc.append("model_decline_risk")
    if not r["has_kw"]:
        rc.append("no_keyword_data")
    if r["avg_position"] <= 0:
        rc.append("no_position_data")
    if not r["has_wc"]:
        rc.append("no_wordcount")
    REASON_LIST.append("|".join(rc) if rc else "no_trigger")

q["reason_codes"] = REASON_LIST

def archetype(r, rs):
    # 1) hard, page-level, actionable signals first
    if "thin_visible_page" in rs:
        return "thin_content_candidate"
    if "low_ctr_visible_page" in rs:
        return "ctr_candidate"
    if "low_engagement_visible_page" in rs:
        return "engagement_candidate"
    if "page_one_decay_risk" in rs or "stale_visible_page" in rs or "declining_with_demand" in rs:
        return "refresh_candidate"
    # 2) visible-but-under-page-one pages with real demand and decent click-through
    if 10 < r["avg_position"] <= 50 and r["impressions_90d"] >= 500 and r["ctr"] >= 0.4 and r["trend_direction"] != "down":
        return "ranking_candidate"
    # 3) evidence gaps -> human, never auto
    if "no_position_data" in rs or "no_keyword_data" in rs or "no_wordcount" in rs:
        return "insufficient_data"
    # 4) model score only
    if "model_decline_risk" in rs:
        return "monitor_candidate"
    return "healthy"

arch = []
for _, r in q.iterrows():
    arch.append(archetype(r, set(r["reason_codes"].split("|"))))
q["archetype"] = arch

ACTION = {"refresh_candidate": "refresh",
          "thin_content_candidate": "expand_and_refresh",
          "ctr_candidate": "refresh_and_review_ctr",
          "engagement_candidate": "refresh_and_review_engagement",
          "ranking_candidate": "recheck_position",
          "monitor_candidate": "monitor",
          "insufficient_data": "human_review",
          "healthy": "monitor"}
q["action"] = [ACTION[at] for at in q["archetype"]]
q["priority"] = ["high" if p >= 0.65 else ("medium" if p >= 0.50 else "low")
                 for p in q["model_probability"]]
print("Priority is a PROPOSED label from the out-of-fold probability (0.65 / 0.50 thresholds).\n")

q = q.sort_values(["model_probability", "impressions_90d"], ascending=[False, False]).reset_index(drop=True)
q["rank"] = q.index + 1

COLUMNS = ["rank", "content_id", "client_id", "model_probability", "priority", "archetype",
           "action", "reason_codes", "decline_label",
           "impressions_90d", "clicks_90d", "sessions_90d", "avg_position", "ctr",
           "content_age_days", "days_since_last_update", "word_count",
           "trend_direction", "content_type", "main_intent"]
queue = q[COLUMNS].copy()

print("Queue rows:", len(queue), "(every content item scored)")
print("\n  action counts\n", queue["action"].value_counts().to_string())
print("\n  archetype counts\n", queue["archetype"].value_counts().to_string())
print("\n  priority counts\n", queue["priority"].value_counts().to_string())

print("\nTop-15 of the ranked queue (what a content person gets first):")
queue.head(15)

Priority is a PROPOSED label from the out-of-fold probability (0.65 / 0.50 thresholds).

Queue rows: 30000 (every content item scored)

  action counts
 action
refresh_and_review_ctr           9741
refresh                          7645
monitor                          5040
human_review                     3935
refresh_and_review_engagement    3415
recheck_position                  142
expand_and_refresh                 82

  archetype counts
 archetype
ctr_candidate             9741
refresh_candidate         7645
insufficient_data         3935
engagement_candidate      3415
monitor_candidate         2933
healthy                   2107
ranking_candidate          142
thin_content_candidate      82

  priority counts
 priority
medium    10905
low       10728
high       8367

Top-15 of the ranked queue (what a content person gets first):


,rank,content_id,client_id,model_probability,priority,archetype,action,reason_codes,decline_label,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,content_age_days,days_since_last_update,word_count,trend_direction,content_type,main_intent
0,1,content_4560b0a818ab,client_19581e27de,1.000000,high,ctr_candidate,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,0,4238,9,4345,7.9,0.21,421,22,NaN,stable,keyword article,commercial
1,2,content_14af58873e4f,client_349c41201b,0.999500,high,engagement_candidate,refresh_and_review_engagement,low_engagement_visible_page|model_decline_risk,0,163479,964,1554,6.5,0.59,124,20,2983.0,up,keyword article,transactional
2,3,content_24736e7d70a7,client_6208ef0f77,0.994726,high,engagement_candidate,refresh_and_review_engagement,declining_with_demand|low_engagement_visible_p...,1,4577,25,1276,16.4,0.55,118,20,3677.0,down,keyword article,informational
3,4,content_cb8aca22fc44,client_6208ef0f77,0.990093,high,engagement_candidate,refresh_and_review_engagement,low_engagement_visible_page|model_decline_risk,0,15176,280,635,8.2,1.85,104,20,3551.0,up,keyword article,informational
4,5,content_35e6edbb0a2d,client_349c41201b,0.987882,high,engagement_candidate,refresh_and_review_engagement,low_engagement_visible_page|model_decline_risk,0,3115,5,1400,23.7,0.16,144,20,2658.0,stable,keyword article,informational
5,6,content_a19c8ee994c2,client_349c41201b,0.972878,high,engagement_candidate,refresh_and_review_engagement,low_engagement_visible_page|model_decline_risk,0,124782,791,1091,3.0,0.63,144,20,5845.0,up,keyword article,informational
6,7,content_d0513fb2a904,client_6208ef0f77,0.930450,high,engagement_candidate,refresh_and_review_engagement,low_engagement_visible_page|model_decline_risk,0,4293,14,1044,21.4,0.33,125,104,3548.0,up,keyword article,informational
7,8,content_a05d66a41827,client_6208ef0f77,0.919489,high,engagement_candidate,refresh_and_review_engagement,declining_with_demand|page_one_decay_risk|low_...,1,57078,1324,1854,5.1,2.32,286,104,7298.0,down,keyword article,informational
8,9,content_2cb567c3c89b,client_6208ef0f77,0.912845,high,engagement_candidate,refresh_and_review_engagement,low_engagement_visible_page|model_decline_risk,0,497727,487,910,22.2,0.10,153,48,6183.0,up,keyword article,informational
9,10,content_a9df8b8a5b1e,client_6208ef0f77,0.902546,high,engagement_candidate,refresh_and_review_engagement,low_engagement_visible_page|model_decline_risk,0,8714,27,452,20.3,0.31,104,8,3442.0,up,keyword article,informational


### 1.3 Decay / refresh insight -- what is measured vs what is *proposed*

**Observed (measured in this snapshot):** `days_since_last_update` is heavily right-skewed -- most items
are recent. The code cell below prints decline rate by recency bucket and the count of long-stale items.
That is an *association in a single snapshot*, not an experiment.

**Not measured (state plainly):** there is **no refresh-event column** and **no before/after design** in
this repo. We therefore cannot claim refreshing *causes* recovery. The model only ranks which items most
*look* declining, from a 30-vs-prev-30-day trend label.

**Proposed -- conservative refresh rule (decision support, not a promise):**
- Refresh candidates are rows with `refresh_candidate` archetype **and** human sign-off.
- **No auto-refresh, no auto-publish.** Every refresh is an editorial act reviewed in Section 3.

In [3]:
# ================= 1c. The freshness/decay evidence that IS in the data ==================
df["fresh_bucket"] = pd.cut(df["days_since_last_update"],
                            bins=[-1, 30, 90, 180, 365, 10e9],
                            labels=["0-30d", "31-90d", "91-180d", "181-365d", "365d+"])
fresh = (df.groupby("fresh_bucket", observed=True)
          .agg(n_items=("decline_label", "size"),
               decline_rate=("decline_label", "mean"),
               median_impressions=("impressions_90d", "median"))
          .reset_index())
fresh["fresh_bucket"] = fresh["fresh_bucket"].astype(str)
fresh["decline_rate"] = fresh["decline_rate"].round(3)
print("Decline rate by recency of last update (measured, cross-sectional):")
print(fresh.to_string(index=False))
print("\nlong-stale items (last update >= 180 days ago):", int((df['days_since_last_update'] >= 180).sum()))

refresh_sig = (
    ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500))
    | ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 100))
    | ((df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["content_age_days"] >= 180))
)
print("\nshare of items carrying a refresh-target signal (any of the 3 refresh reasons): %.1f%%"
      % (100 * refresh_sig.mean()))

Decline rate by recency of last update (measured, cross-sectional):
fresh_bucket  n_items  decline_rate  median_impressions
       0-30d    20480         0.511               470.0
      31-90d      175         0.589               510.0
     91-180d     9171         0.611              1692.0
    181-365d      169         0.467                16.0
       365d+        5         0.600                 2.0

long-stale items (last update >= 180 days ago): 174

share of items carrying a refresh-target signal (any of the 3 refresh reasons): 58.0%


## 2. Intended use and limits

**Intended use.** A content human (editor / SEO analyst / content operator) uses this queue as a
**decision-support ordering of what to review first** -- *not* an instruction engine. It answers one
question: *"which content pages should get next week's review effort, and why?"* -- with a reason code and
an archetype on every row so the answer can be checked against the raw page metrics.

**What the queue is NOT -- limits (each verified below in code, no invented values):**
1. **Not a per-page classifier you act on without sign-off.** The fold-level precision spread is wide;
   the queue is a directional *orderer*, not a guarantee.
2. **Not a causal statement.** Nothing here shows that refreshing *produces* a recovery.
3. **Not cross-time generalisable.** One snapshot; trailing-90d + 30d window columns; no seasonality;
   no refresh-event column; no outcome feedback.
4. **Not production.** There is no serving, monitoring, drift guard, or retraining in this repo. It is an
   analysis artifact for the paper, regenerated on demand.

In [4]:
# ================== 2.a Limits with numbers measured in this dataset =====================
limits = pd.DataFrame({
    "limit": [
        "thin evidence: impressions_90d < 100",
        "'no position' rows (avg_position == 0 -> no data)",
        "missing keyword context (search_volume is NaN)",
        "missing word_count",
        "declining rows (label count)",
        "content items per fold (avg)",
    ],
    "measured": [
        int((df["impressions_90d"] < 100).sum()),
        int((df["avg_position"] == 0).sum()),
        int(df["search_volume"].isna().sum()),
        int(df["word_count"].isna().sum()),
        int(df["decline_label"].sum()),
        int(round(len(df) / 5)),
    ],
})
print(limits.to_string(index=False))
print("\n  observed base rate: %.3f" % df["decline_label"].mean())

                                            limit  measured
             thin evidence: impressions_90d < 100      7994
'no position' rows (avg_position == 0 -> no data)      1205
   missing keyword context (search_volume is NaN)      2468
                               missing word_count      7699
                     declining rows (label count)     16262
                     content items per fold (avg)      6000

  observed base rate: 0.542


### Cost / value thinking -- qualitative framework (clearly *proposed*, not measured $)

There is **no cost or effort data in this repo**, so this is a decision rule, not a claim:

| Archetype | Value frame (why it's worth the human hour) | Effort (human, qualitative) | Review order |
|---|---|---|---|
| `ctr_candidate` | shows in SERP but under-clicks: small edit may lift the one metric | low (title/match-intent) | **1st** |
| `thin_content_candidate` | visible but short: depth may cap positioning | medium (rewrite) | 2nd |
| `refresh_candidate` | declining **with** visible demand | medium (refresh) | 3rd |
| `engagement_candidate` | visible but losing sessions | medium | 4th |
| `ranking_candidate` | visible below page one, decent CTR | medium (position audit) | 5th |
| `monitor_candidate` | model says risk, no concrete page signal yet | low (watch-list) | in queue |
| `insufficient_data` | no position/keyword evidence to justify action | review manually / skip | last |

Rule: **score does not set priority by itself.** We take the cheapest high-confidence visible fix first
(CTR -> thin -> refresh) and let effort act as a tie-breaker inside the same priority band.

## 3. Human review + the no-go list

**Before any action is taken on any row, a human reviewer must hold each check (rule, not optional):**

1. **Search intent** -- does the intent match the page's purpose? A *mismatch* is a quick meta fix, not a refresh.
2. **Content quality** -- is the page genuinely poor, or just mispositioned?
3. **Current SERP context** -- is the page actually ranking for the query that shows the clicks?
4. **Traffic trend** -- the decline label is *a single number*; check the trend before believing it.
5. **Business relevance** -- is this item still commercial/navigational? Deprecated/halo pages leave the queue, not "get refreshed".
6. **Data completeness** -- does the row carry a `no_position_data` / `no_keyword_data` / `no_wordcount` code? If so -> `insufficient_data`, not an action.

**The reviewer may always REJECT.** A recommendation is a starting point, never an order.

In [5]:
# ============== 3.a. Preview = a reviewer opens + the "never automate" list ==============
preview_cols = ["rank", "archetype", "reason_codes", "impressions_90d", "avg_position", "ctr",
                "sessions_90d", "days_since_last_update", "trend_direction", "word_count"]
print("First 8 rows of the queue as the reviewer sees them:")
batch = queue[preview_cols].head(8).copy()
batch.insert(0, "review?", "____")
print(batch.to_string(index=False))

NO_GO = [
    "automatically publishing, deleting or redirecting content items",
    "automatically changing titles, meta descriptions, canonical or schema tags",
    "acting on the model score alone, without the human checklist above",
    "treating refresh == fix (no causal claim anywhere here)",
    "acting on rows with no_position_data / no_keyword_data / no_wordcount (insufficient evidence)",
    "letting score alone decide priority -- effort & intent always act as tie-breakers",
    "turning this queue into an auto-recommender for the portfolio or a public product",
]
print("\nWHAT MUST **NOT** BE AUTOMATED (non-negotiable):")
for i, msg in enumerate(NO_GO, 1):
    print(f"  {i}. {msg}")
print("\nCore principle: the model recommends; the human decides.")

First 8 rows of the queue as the reviewer sees them:
review?  rank            archetype                                                                                         reason_codes  impressions_90d  avg_position  ctr  sessions_90d  days_since_last_update trend_direction  word_count
   ____     1        ctr_candidate page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page|model_decline_risk|no_wordcount             4238           7.9 0.21          4345                      22          stable         NaN
   ____     2 engagement_candidate                                                       low_engagement_visible_page|model_decline_risk           163479           6.5 0.59          1554                      20              up      2983.0
   ____     3 engagement_candidate                                 declining_with_demand|low_engagement_visible_page|model_decline_risk             4577          16.4 0.55          1276                      20            down      36

## 4. Monitoring / retrain triggers

**Observed (measured today):** AUC mean +/- std across the 5 client folds; precision at the head of the
queue; the decline base rate. Exact figures printed in the cell below.

**Proposed** (= **governance rules**, clearly *not* measured): we have **no** production telemetry, drift,
or retraining history in this repo. The triggers below are what the team *should* watch -- thresholds are
proposed values for periodic review, not claims that any of them has fired.

In [6]:
# ================= 4. Observed values + proposed monitoring / retrain triggers ===========
observed = pd.DataFrame({
    "metric": [
        "out-of-fold AUC (mean of folds)",
        "queue top-50 precision (measured)",
        "decline base rate (whole snapshot)",
        "rows marked model_decline_risk (P>=0.50)",
    ],
    "value": [
        "%.3f +/- %.3f" % (folds["AUC"].mean(), folds["AUC"].std()),
        "%.3f (base %.3f)" % (queue["decline_label"].head(50).mean(), y.mean()),
        "%.3f" % y.mean(),
        str(int((df["model_probability"] >= 0.50).sum())),
    ],
})
print("OBSERVED - measured in this run:")
print(observed.to_string(index=False))

proposed = pd.DataFrame({
    "monitor / retrain trigger": [
        "AUC across a fresh client-grouped 5-fold re-validation",
        "shape of the model_probability distribution",
        "missing-rate on keyword / position / word_count columns",
        "reviewer acceptance vs override rate on the queue",
        "queue rank stability between score re-runs",
        "definition change of the 'declining' label",
        "sufficient new outcome labels available to the fold set",
    ],
    "sign the playbook is stale (proposed threshold)": [
        "AUC mean < ~0.55 on two consecutive re-validation runs",
        "distribution shift > ~2 sd of the seeded scoring run",
        "if the no-evidence rows jump sharply (data pipeline issue)",
        "> ~20% of the top reviewed actions get overridden",
        "top-500 rank correlation < ~0.6 vs the prior run",
        "any change to how 'declining' is defined -> adjust labels + retrain",
        "retrain whenever the fold composition materially changes",
    ],
})
print("\nPROPOSED monitoring / retrain triggers (governance; not claims of having fired):")
print(proposed.to_string(index=False))

OBSERVED - measured in this run:
                                  metric              value
         out-of-fold AUC (mean of folds)    0.632 +/- 0.037
       queue top-50 precision (measured) 0.640 (base 0.542)
      decline base rate (whole snapshot)              0.542
rows marked model_decline_risk (P>=0.50)              19272

PROPOSED monitoring / retrain triggers (governance; not claims of having fired):
                              monitor / retrain trigger                     sign the playbook is stale (proposed threshold)
 AUC across a fresh client-grouped 5-fold re-validation              AUC mean < ~0.55 on two consecutive re-validation runs
            shape of the model_probability distribution                distribution shift > ~2 sd of the seeded scoring run
missing-rate on keyword / position / word_count columns          if the no-evidence rows jump sharply (data pipeline issue)
      reviewer acceptance vs override rate on the queue                   > ~20% of the t

## 5. Exports for the paper

*Outputs below are regenerated fresh every run.*

- **`work/outputs/ranked_action_queue.csv`** -- the full 30,000-row ranked queue (rank, reason codes,
  actions, key page metrics). **Intentionally NOT committed**: `work/**/*.csv` is gitignored and CI
  blocks data CSVs -- this notebook recreates it on every execution.
- **Figures** -> `work/figures/` (reusable for the draft): queue-head precision vs base rate, archetype
  mix, score distribution, freshness decline rate.
- **Nothing else is written** to the repo by this notebook.

In [7]:
# ======================== 5. Export queue CSV and paper figures =========================
import os
os.makedirs("../outputs", exist_ok=True)
os.makedirs("../figures", exist_ok=True)
QUEUE_PATH = Path("../outputs/ranked_action_queue.csv")
queue.to_csv(QUEUE_PATH, index=False)
print("Wrote queue:", QUEUE_PATH, "| rows:", len(queue))

def save(fig, name):
    fig.savefig(Path("../figures") / name, format="svg", bbox_inches="tight")

# --- fig: precision at the head of the queue vs base rate ---
head = pd.DataFrame({
    "k": ["top-20", "top-50", "top-100"],
    "precision": [round(queue["decline_label"].head(20).mean(), 3),
                  round(queue["decline_label"].head(50).mean(), 3),
                  round(queue["decline_label"].head(100).mean(), 3)],
})
head["base_rate"] = round(y.mean(), 3)
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.bar(head["k"], head["precision"], color="#4E79A7")
ax.axhline(head["base_rate"].iloc[0], color="grey", linestyle="--", label="base rate")
ax.set_ylabel("precision")
ax.set_title("Precision at the queue head (out-of-fold scores)")
ax.legend()
save(fig, "fig_precision_head.svg"); plt.close(fig)

# --- fig: action mix ---
fig2, ax2 = plt.subplots(figsize=(9, 4))
q["action"].value_counts().sort_values().plot(kind="barh", ax=ax2, color="#59A14F")
ax2.set_title("Proposed action labels across the queue")
ax2.set_xlabel("items")
save(fig2, "fig_action_mix.svg"); plt.close(fig2)

# --- fig: archetype mix ---
fig3, ax3 = plt.subplots(figsize=(9, 4))
q["archetype"].value_counts().sort_values().plot(kind="barh", ax=ax3, color="#E15759")
ax3.set_title("Archetype mix (whole snapshot)")
ax3.set_xlabel("items")
save(fig3, "fig_archetype_mix.svg"); plt.close(fig3)

# --- fig: out-of-fold score distribution ---
fig4, ax4 = plt.subplots(figsize=(8, 4))
ax4.hist(df["model_probability"], bins=40, color="#B07AA1", edgecolor="white")
ax4.axvline(0.50, color="grey", linestyle=":", label="model_decline_risk threshold")
ax4.set_title("Out-of-fold P(decline) distribution")
ax4.set_xlabel("model_probability"); ax4.set_ylabel("items"); ax4.legend()
save(fig4, "fig_score_distribution.svg"); plt.close(fig4)

# --- fig: freshness recency ---
fig5, ax5 = plt.subplots(figsize=(8, 4))
ax5.bar(fresh["fresh_bucket"], fresh["decline_rate"], color="#76B7B2")
ax5.set_title("Decline rate by time since last update (cross-sectional)")
ax5.set_xlabel("days since last update"); ax5.set_ylabel("observed share declining")
save(fig5, "fig_freshness_rate.svg"); plt.close(fig5)

print("\nFigures written to work/figures/:")
for f in sorted(Path("../figures").glob("fig_*.svg")):
    print("   ", f.name, " (%.1f KB)" % (f.stat().st_size / 1000))

Wrote queue: ..\outputs\ranked_action_queue.csv | rows: 30000



Figures written to work/figures/:
    fig_action_mix.svg  (36.3 KB)
    fig_archetype_mix.svg  (38.0 KB)
    fig_freshness_rate.svg  (34.8 KB)
    fig_precision_head.svg  (29.0 KB)
    fig_score_distribution.svg  (42.8 KB)


## Self-check

- [x] Section 1: ranked queue exists (30,000 rows, out-of-fold scores)
- [x] Section 1: reason codes on every row (re-derivable from data columns)
- [x] Archetype -> action mapping present, one archetype per row
- [x] Decay/refresh insight distinguishes **measured** vs **proposed**
- [x] Intended use + limits stated with measured numbers
- [x] Cost / value 2x2 included (qualitative, clearly not $)
- [x] Human review rules + what-must-not-be-automated
- [x] Monitoring / retrain triggers: observed vs proposed separated
- [x] Queue + figures exported to `work/outputs/` and `work/figures/`

In [8]:
# ==================== 6. Verified: automated success checks ===========================
ok = True
def check(name, condition):
    global ok
    ok = ok and bool(condition)
    print(("PASS" if condition else "FAIL"), " |", name)
    return ok

check("queue rows == 30,000", len(queue) == 30000)
check("ranks are 1..N contiguous", list(queue["rank"]) == list(range(1, len(queue) + 1)))
check("every row has an out-of-fold score", queue["model_probability"].notna().all())
check("every row has reason codes", (queue["reason_codes"].str.len() > 0).all())
check("every row has an archetype + action", ((queue["archetype"] != "") & (queue["action"] != "")).all())
check("queue CSV exported + non-empty", QUEUE_PATH.exists() and QUEUE_PATH.stat().st_size > 0)
check("figures exported (>=5 svg)", len(list(Path("../figures").glob("fig_*.svg"))) >= 5)
check("no URLs / private strings in exported columns",
      not any(x.str.contains("http").any()
              for x in [queue[c].astype(str) for c in ["content_id", "client_id", "content_type", "main_intent"]]))
print("\nOVERALL: %s" % ("PASS" if ok else "FAIL - fix before submitting"))

PASS  | queue rows == 30,000
PASS  | ranks are 1..N contiguous
PASS  | every row has an out-of-fold score
PASS  | every row has reason codes
PASS  | every row has an archetype + action
PASS  | queue CSV exported + non-empty
PASS  | figures exported (>=5 svg)
PASS  | no URLs / private strings in exported columns

OVERALL: PASS
